In [157]:
import numpy as np
import pickle
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline


In [158]:
#read the dataset
df = pd.read_csv("dataset/Cars.csv")
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [159]:
# Check for duplicates in the dataset
df.duplicated().sum()

np.int64(1202)

In [160]:
# View the duplicate rows to understand which entries are repeated and if they need to be addressed (e.g., by removing or consolidating them).
df[df.duplicated(keep=False)].sort_values(
    by=["name", "year"]
).head(20)

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
1977,Audi Q3 2.0 TDI Quattro Premium Plus,2017,2825000,22000,Diesel,Dealer,Automatic,First Owner,15.73 kmpl,1968 CC,174.33 bhp,380Nm@ 1750-2500rpm,5.0
7324,Audi Q3 2.0 TDI Quattro Premium Plus,2017,2825000,22000,Diesel,Dealer,Automatic,First Owner,15.73 kmpl,1968 CC,174.33 bhp,380Nm@ 1750-2500rpm,5.0
2129,Audi Q5 3.0 TDI Quattro,2014,1850000,76131,Diesel,Individual,Automatic,First Owner,13.22 kmpl,2967 CC,241.4 bhp,580Nm@ 1400-3250rpm,5.0
7775,Audi Q5 3.0 TDI Quattro,2014,1850000,76131,Diesel,Individual,Automatic,First Owner,13.22 kmpl,2967 CC,241.4 bhp,580Nm@ 1400-3250rpm,5.0
131,Audi Q5 35TDI Premium Plus,2018,3975000,31800,Diesel,Dealer,Automatic,First Owner,17.01 kmpl,1968 CC,188 bhp,400nm@ 1750-3000rpm,5.0
1561,Audi Q5 35TDI Premium Plus,2018,3975000,31800,Diesel,Dealer,Automatic,First Owner,17.01 kmpl,1968 CC,188 bhp,400nm@ 1750-3000rpm,5.0
1857,Audi Q5 35TDI Premium Plus,2018,3975000,31800,Diesel,Dealer,Automatic,First Owner,17.01 kmpl,1968 CC,188 bhp,400nm@ 1750-3000rpm,5.0
3236,Audi Q5 35TDI Premium Plus,2018,3975000,31800,Diesel,Dealer,Automatic,First Owner,17.01 kmpl,1968 CC,188 bhp,400nm@ 1750-3000rpm,5.0
5245,Audi Q5 35TDI Premium Plus,2018,3975000,31800,Diesel,Dealer,Automatic,First Owner,17.01 kmpl,1968 CC,188 bhp,400nm@ 1750-3000rpm,5.0
7699,Audi Q5 35TDI Premium Plus,2018,3975000,31800,Diesel,Dealer,Automatic,First Owner,17.01 kmpl,1968 CC,188 bhp,400nm@ 1750-3000rpm,5.0


In [161]:
#Since they are completely identical, we can drop the duplicates to avoid biasing our model with repeated data points.
df = df.drop_duplicates()

In [162]:
# Check the distribution of the 'owner' column
df['owner'].value_counts()

owner
First Owner             4242
Second Owner            1974
Third Owner              536
Fourth & Above Owner     169
Test Drive Car             5
Name: count, dtype: int64

In [163]:
# Map the 'owner' column to numerical values
owner_mapping = {
    "First Owner": 1,
    "Second Owner": 2,
    "Third Owner": 3,
    "Fourth & Above Owner": 4,
    "Test Drive Car": 5
}

df["owner"] = df["owner"].map(owner_mapping)

In [164]:
#display the first few rows of the DataFrame to verify the changes
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,1,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,2,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,3,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,1,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,1,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [165]:
# Check the distribution of the 'owner' column to ensure the mapping was successful
df['owner'].value_counts()

owner
1    4242
2    1974
3     536
4     169
5       5
Name: count, dtype: int64

In [166]:
# Check the unique values in the 'fuel' column
df['fuel'].unique()

<ArrowStringArray>
['Diesel', 'Petrol', 'LPG', 'CNG']
Length: 4, dtype: str

In [167]:
# Remove rows with specific fuel types, as they use a different mileage system
remove_values = ["CNG", "LPG"]
df = df[~df["fuel"].isin(remove_values)]

In [168]:
# Check the unique values in the 'fuel' column after removing the specified fuel types
df['fuel'].unique()

<ArrowStringArray>
['Diesel', 'Petrol']
Length: 2, dtype: str

In [169]:
# Check the first few rows of the 'mileage' column and its data type
print(df["mileage"].head())
print(df["mileage"].dtype)

0     23.4 kmpl
1    21.14 kmpl
2     17.7 kmpl
3     23.0 kmpl
4     16.1 kmpl
Name: mileage, dtype: str
str


In [170]:
# Check for missing values in the 'mileage' column
df["mileage"].isna().sum()

np.int64(201)

In [171]:
# Display rows with missing values in the 'mileage' column
# do not handle the NaN value yet. just display the rows with NaN values in the mileage column to understand the data better.
df[df["mileage"].isnull()]

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
13,Maruti Swift 1.3 VXi,2007,200000,80000,Petrol,Individual,Manual,2,NaN,NaN,NaN,NaN,NaN
31,Fiat Palio 1.2 ELX,2003,70000,50000,Petrol,Individual,Manual,2,NaN,NaN,NaN,NaN,NaN
78,Tata Indica DLS,2003,50000,70000,Diesel,Individual,Manual,1,NaN,NaN,NaN,NaN,NaN
87,Maruti Swift VDI BSIV W ABS,2015,475000,78000,Diesel,Dealer,Manual,1,NaN,NaN,NaN,NaN,NaN
119,Maruti Swift VDI BSIV,2010,300000,120000,Diesel,Individual,Manual,2,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7740,Hyundai Santro Xing XG,2004,70000,70000,Petrol,Individual,Manual,2,NaN,NaN,NaN,NaN,NaN
7996,Hyundai Santro LS zipPlus,2000,140000,50000,Petrol,Individual,Manual,2,NaN,NaN,NaN,NaN,NaN
8009,Hyundai Santro Xing XS eRLX Euro III,2006,145000,80000,Petrol,Individual,Manual,2,NaN,NaN,NaN,NaN,NaN
8068,Ford Figo Aspire Facelift,2017,580000,165000,Diesel,Individual,Manual,1,NaN,NaN,NaN,NaN,NaN


In [172]:
# Check the shape of the DataFrame
df.shape

(6832, 13)

In [173]:
# Convert the 'mileage' column to float by extracting the numeric part and converting it to float
df["mileage"] = df.mileage.str.split().str[0].astype(float)

In [174]:
# Check the first few rows of the 'mileage' column and its data type to see if the conversion was successful
print(df["mileage"].head())
print(df["mileage"].dtype)

0    23.40
1    21.14
2    17.70
3    23.00
4    16.10
Name: mileage, dtype: float64
float64


In [175]:
# Convert the 'engine' column to float by extracting the numeric part and converting it to float
df["engine"] = df.engine.str.split().str[0].astype(float)

In [176]:
# Check the first few rows of the 'engine' column and its data type to see if the conversion was successful
print(df["engine"].head())
print(df["engine"].dtype)

0    1248.0
1    1498.0
2    1497.0
3    1396.0
4    1298.0
Name: engine, dtype: float64
float64


In [177]:
# Convert the 'max_power' column to float by extracting the numeric part and converting it to float
df["max_power"] = df["max_power"].str.split().str[0].astype(float)

In [178]:
#  Check the first few rows of the 'max_power' column and its data type to see if the conversion was successful
print(df["max_power"].head())
print(df["max_power"].dtype)

0     74.00
1    103.52
2     78.00
3     90.00
4     88.20
Name: max_power, dtype: float64
float64


In [179]:
# Check the first few rows of the 'name' column and its data type

print(df["name"].head())
print(df["name"].dtype)

0          Maruti Swift Dzire VDI
1    Skoda Rapid 1.5 TDI Ambition
2        Honda City 2017-2020 EXi
3       Hyundai i20 Sportz Diesel
4          Maruti Swift VXI BSIII
Name: name, dtype: str
str


In [180]:
# Split the 'name' column to extract the first word (brand name) and assign it back to the 'name' column
df["name"] = df["name"].str.split().str[0]

In [181]:
# Check the first few rows of the 'name' column and its data type to see if the conversion was successful
print(df["name"].head())
print(df["name"].dtype)

0     Maruti
1      Skoda
2      Honda
3    Hyundai
4     Maruti
Name: name, dtype: object
object


In [182]:
# Drop the 'torque' column from the DataFrame, as it may not be relevant for the analysis or model training
df = df.drop("torque", axis=1)

In [183]:
# Check the columns of the DataFrame to verify the changes
df.columns

Index(['name', 'year', 'selling_price', 'km_driven', 'fuel', 'seller_type',
       'transmission', 'owner', 'mileage', 'engine', 'max_power', 'seats'],
      dtype='str')

In [184]:
# Remove rows where the 'owner' column has a value of 5 (Test Drive Car), as they are ridiculously expensive
df = df[df["owner"] != 5]

In [185]:
# Check the distribution of the 'owner' column after removing the rows with 'Test Drive Car' to ensure the changes were successful
df['owner'].value_counts()

owner
1    4191
2    1943
3     528
4     165
Name: count, dtype: int64

In [186]:
# Check for any remaining missing values in the DataFrame
df.isnull().sum()

name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          201
engine           201
max_power        198
seats            201
dtype: int64

In [187]:
# Check the shape of the DataFrame after all the preprocessing steps
df.shape

(6827, 12)

In [188]:
#There are 8,028 rows (samples) and 12 columns (features + target)
#We have 214 rows with missing values in some columns, which is only about 2.7% of the dataset. 
# I think it is reasonable to remove these rows since we would still retain about 97.3% of the data.
df = df.dropna()

In [189]:
#check if there are any missing values left
df.isnull().sum()

name             0
year             0
selling_price    0
km_driven        0
fuel             0
seller_type      0
transmission     0
owner            0
mileage          0
engine           0
max_power        0
seats            0
dtype: int64

In [190]:
# Since selling_price is a big number, it can cause your prediction to be very unstable
# transform the target variable 'selling_price' using logarithm to reduce skewness and make it more normally distributed
y = np.log(df['selling_price'])

#drop the target variable from the feature set
X = df.drop("selling_price", axis=1)

In [191]:
#Split the dataset into training and testing sets, with 80% of the data used for training and 20% for testing. The random_state parameter is set to 42 to ensure reproducibility of the results.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [192]:
# Check the first few rows of the training set to ensure that the split was successful and the data looks as expected
X_train.head()

,name,year,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats
3890,Maruti,1995,70000,Petrol,Individual,Manual,1,16.10,796.0,37.0,4.0
6291,Maruti,2017,76000,Diesel,Dealer,Manual,1,25.47,1248.0,88.5,7.0
4492,Toyota,2015,190000,Diesel,Individual,Manual,1,12.99,2494.0,100.6,7.0
6794,Tata,2012,120000,Diesel,Individual,Manual,1,23.03,1396.0,69.0,5.0
2786,Tata,2015,120000,Diesel,Individual,Manual,2,19.10,1405.0,70.0,5.0


In [193]:
# Define the categorical features that need to be one-hot encoded
categorical_features = [
    "name",
    "fuel",
    "seller_type",
    "transmission"
]

# Create a ColumnTransformer to apply one-hot encoding to the categorical features while leaving the other features unchanged
# It is needed since the model cannot handle categorical features directly, and one-hot encoding is a common technique to convert categorical variables into a format that can be provided to machine learning algorithms to do a better job in prediction.
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

In [ ]:
# Create a pipeline that first applies the preprocessor to the data and then fits a Linear Regression model to the transformed data
model_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Fit the model to the training data
model_lr.fit(X_train, y_train)

#Predict the target variable for the test set using the trained model
pred_y = model_lr.predict(X_test)
# Convert back to original price
pred_y_original = np.exp(pred_y)
y_test_original = np.exp(y_test)

# Evaluate the model using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R-squared (R²) metrics to assess the performance of the model on the test set.
mae = mean_absolute_error(y_test_original, pred_y_original)
rmse = np.sqrt(mean_squared_error(y_test_original, pred_y_original))
r2 = r2_score(y_test_original, pred_y_original)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


Mean Absolute Error: 104429.67223869752


In [ ]:
# Now, let's train a Decision Tree Regressor model using the same pipeline structure. The Decision Tree model can capture non-linear relationships in the data, which may improve prediction accuracy compared to linear regression.
model_dt = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=42))
])

model_dt.fit(X_train, y_train)
pred_y = model_dt.predict(X_test)

pred_y_original = np.exp(pred_y)
y_test_original = np.exp(y_test)

# Evaluate
mae = mean_absolute_error(y_test_original, pred_y_original)
rmse = np.sqrt(mean_squared_error(y_test_original, pred_y_original))
r2 = r2_score(y_test_original, pred_y_original)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


Mean Absolute Error: 100555.788514969
MAE: 100555.788514969
RMSE: 211461.45943772304
R²: 0.8233336209352339


In [ ]:
# Now, let's train a Random Forest Regressor model using the same pipeline structure. The Random Forest model is an ensemble method that combines multiple decision trees to improve prediction accuracy and reduce overfitting.
model_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
         n_estimators=200,
         max_depth=15,
         min_samples_split=5,
         min_samples_leaf=2,
         random_state=42
    ))
])

model_rf.fit(X_train, y_train)

pred_y = model_rf.predict(X_test)

pred_y_original = np.exp(pred_y)
y_test_original = np.exp(y_test)

# Evaluate
mae = mean_absolute_error(y_test_original, pred_y_original)

mae = mean_absolute_error(y_test_original, pred_y_original)
rmse = np.sqrt(mean_squared_error(y_test_original, pred_y_original))
r2 = r2_score(y_test_original, pred_y_original)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


Mean Absolute Error: 74522.24395682968
MAE: 74522.24395682968
RMSE: 135590.5790639911
R²: 0.9273642394391253


Report

The results show that the Random Forest Regressor performed the best among the three algorithms tested. It achieved an MAE of 74,522.24, RMSE of 135,590.58, and R² of 0.9274 on the test set. This means that the Random Forest predictions differ from the actual selling prices by approximately 74,522 on average, while the R² indicates that about 92.74% of the variation in car selling prices is explained by the model. Linear Regression achieved an MAE of 104,429.67, RMSE of 203,179.51, and R² of 0.8369, while Decision Tree performed slightly worse, with an MAE of 100,555.79, RMSE of 211,461.46, and R² of 0.8233. Random Forest performed better because it combines multiple decision trees and can capture complex, non-linear relationships between car characteristics and selling price, whereas Linear Regression is more limited by its assumption of a linear relationship. The single Decision Tree can also be sensitive to the particular training data and may not generalize as well as an ensemble of trees.

Among the features, car brand/name, year, km_driven, engine, max_power, mileage, fuel, transmission, seller type, owner, and seats were used to predict the selling price. Features such as year, brand/name, km_driven, engine, and max_power are expected to have strong relationships with price because newer cars, well-known brands, less-driven vehicles, and cars with larger or more powerful engines generally have different market values. The fuel, transmission, and seller_type features can also affect price because different fuel types, automatic versus manual transmission, and dealer versus individual sales can correspond to different market segments. On the other hand, seats and owner count may have less influence compared with characteristics such as brand, year, and engine performance. However, the current notebook does not calculate feature-importance values, so the exact ranking of feature importance cannot be claimed from the results alone. The torque feature was removed as instructed, and Test Drive Cars, missing-value records, and duplicate records were also removed during preprocessing.

Overall, Random Forest Regressor is the most suitable model among the three tested models for this dataset. The use of log transformation on selling_price helps reduce the effect of extremely high-priced cars and makes the prediction more stable. Removing duplicate records before the train/test split also makes the evaluation more reliable. The Random Forest model still performs better on the training data than on the test data, indicating some degree of overfitting, but its test R² of 0.9274 shows that it generalizes well to unseen data. Therefore, the Random Forest Regressor was selected as the final model for the car selling-price prediction application.

In [ ]:
from utils import *

In [207]:
#Save the trained Random Forest model to a file using pickle for later use in making predictions on new data. The model is saved in the "models" directory with the filename "car_price_model.pkl".
save("models/car_price_model.pkl", model_rf)

In [ ]:
# Test with sample data prediction using the trained Random Forest model.
new_car = pd.DataFrame({
    "name": ["Maruti Swift"],
    "year": [2018],
    "km_driven": [50000],
    "fuel": ["Diesel"],
    "seller_type": ["Individual"],
    "transmission": ["Manual"],
    "owner": [1],
    "mileage": [24.0],
    "engine": [1248.0],
    "max_power": [74.0],
    "seats": [5]
})

#load the mdoel
l_model_rf = load("models/car_price_model.pkl")

# Predict log(selling_price)
pred_y = l_model_rf.predict(new_car)

# Convert back to original price
predicted_price = np.exp(pred_y[0])

print("Predicted Selling Price:", predicted_price)

Predicted Selling Price: 685533.7558344255


In [209]:
#To see if the model is working correctly, we can perform some checks:
# 1. selling_price should not be in X
print("selling_price" in X.columns)

# 2. Check duplicates
print("Duplicates:", df.duplicated().sum())

# 3. Check train/test sizes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

# 4. Check train and test R²
train_pred = model_rf.predict(X_train)
test_pred = model_rf.predict(X_test)

print("Train R²:", r2_score(y_train, train_pred))
print("Test R²:", r2_score(y_test, test_pred))

False
Duplicates: 19
X_train: (5300, 11)
X_test: (1326, 11)
Train R²: 0.9685721211626587
Test R²: 0.9199267996694266


The Random Forest model performs well, achieving a training R² of 0.969 and a test R² of 0.920, indicating strong predictive performance with some minor overfitting.